In [1]:
import os
os.system('pip install -q pyvi transformers accelerate')

import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from pyvi import ViTokenizer
from tqdm import tqdm

# =========================
# 1. CONFIG
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

class Config:
    model_name = 'vinai/phobert-large'
    max_len = 256
    batch_size = 8
    gradient_accumulation_steps = 2
    epochs = 5
    lr = 1.0e-5 # slightly lower lr for large model
    weight_decay = 0.01
    n_folds = 5
    topic_loss_weight = 0.4
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = Config()

# =========================
# 2. LOAD DATA
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Teencode dict
    replace_dict = {
        'ko': 'không', 'k': 'không', 'khong': 'không', 'kh': 'không', 'khg': 'không',
        'dc': 'được', 'đc': 'được', 'đk': 'được',
        'vs': 'với', 'mk': 'mình', 'mik': 'mình',
        'r': 'rồi', 'cx': 'cũng', 'trc': 'trước',
        'rất': 'rất', 'râk': 'rất', 'bt': 'bình thường',
        'thik': 'thích', 'ok': 'ổn', 'oke': 'ổn'
    }
    for k, v in replace_dict.items():
        text = re.sub(r'\b{}\b'.format(k), v, text)
    return text

try:
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')
except:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

train_df['sentence'] = train_df['sentence'].apply(lambda x: ViTokenizer.tokenize(clean_text(x)))
test_df['sentence'] = test_df['sentence'].apply(lambda x: ViTokenizer.tokenize(clean_text(x)))

# =========================
# 3. DATASET
# =========================
class NLPDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'sentence']
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=cfg.max_len,
            # No padding here, do it dynamically in dataloader
        )

        item = {
            'input_ids': torch.tensor(enc['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long)
        }

        if not self.is_test:
            item['sentiment'] = torch.tensor(self.df.loc[idx, 'sentiment'], dtype=torch.long)
            item['topic'] = torch.tensor(self.df.loc[idx, 'topic'], dtype=torch.long)

        return item

# =========================
# 4. MODEL
# =========================
class SentimentModel(nn.Module):
    def __init__(self, model_name, num_sent=3, num_topic=4):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        
        # Multi-Sample Dropout
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        
        self.sent_head = nn.Linear(hidden, num_sent)
        self.topic_head = nn.Linear(hidden, num_topic)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        
        # Mean Pooling
        last_hidden_state = out.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        # Apply multi-sample dropout
        sent_out = torch.mean(torch.stack([self.sent_head(dropout(mean_pooled)) for dropout in self.dropouts], dim=0), dim=0)
        topic_out = torch.mean(torch.stack([self.topic_head(dropout(mean_pooled)) for dropout in self.dropouts], dim=0), dim=0)
        
        return topic_out, sent_out

# =========================
# 5. FGM
# =========================
class FGM:
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=0.5, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0:
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

# =========================
# 6. TRAIN
# =========================
def train_one_epoch(model, loader, optimizer, scheduler, crit_sent, crit_topic, fgm):
    model.train()
    losses = []

    for step, batch in enumerate(tqdm(loader)):
        input_ids = batch['input_ids'].to(cfg.device)
        attention_mask = batch['attention_mask'].to(cfg.device)
        y_sent = batch['sentiment'].to(cfg.device)
        y_topic = batch['topic'].to(cfg.device)

        topic_logits, sent_logits = model(input_ids, attention_mask)

        loss_sent = crit_sent(sent_logits, y_sent)
        loss_topic = crit_topic(topic_logits, y_topic)
        loss = (loss_sent + cfg.topic_loss_weight * loss_topic) / cfg.gradient_accumulation_steps
        loss.backward()

        # FGM attack
        fgm.attack()
        topic_logits_adv, sent_logits_adv = model(input_ids, attention_mask)
        adv_loss = (crit_sent(sent_logits_adv, y_sent) + cfg.topic_loss_weight * crit_topic(topic_logits_adv, y_topic)) / cfg.gradient_accumulation_steps
        adv_loss.backward()
        fgm.restore()

        if (step + 1) % cfg.gradient_accumulation_steps == 0 or (step + 1) == len(loader):
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        losses.append(loss.item() * cfg.gradient_accumulation_steps)

    return np.mean(losses)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, labels = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(cfg.device)
        attention_mask = batch['attention_mask'].to(cfg.device)
        y_sent = batch['sentiment'].cpu().numpy()

        _, sent_logits = model(input_ids, attention_mask)
        pred = torch.argmax(sent_logits, dim=1).cpu().numpy()

        preds.extend(pred)
        labels.extend(y_sent)

    return f1_score(labels, preds, average='macro')

# =========================
# 7. KFOLD TRAINING
# =========================
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Custom collate for Multi-task Data
def collate_fn(batch):
    batch_out = data_collator([{'input_ids': b['input_ids'], 'attention_mask': b['attention_mask']} for b in batch])
    if 'sentiment' in batch[0]:
        batch_out['sentiment'] = torch.stack([b['sentiment'] for b in batch])
        batch_out['topic'] = torch.stack([b['topic'] for b in batch])
    return batch_out

skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=42)

test_probs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['sentiment'])):
    print(f'===== FOLD {fold+1} =====')

    tr_df = train_df.iloc[train_idx].copy()
    va_df = train_df.iloc[val_idx].copy()

    # weighted sampler
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(tr_df['sentiment']),
        y=tr_df['sentiment']
    )
    weight_map = dict(zip(np.unique(tr_df['sentiment']), class_weights))
    sample_weights = tr_df['sentiment'].map(weight_map).values
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_ds = NLPDataset(tr_df, tokenizer)
    val_ds = NLPDataset(va_df, tokenizer)
    test_ds = NLPDataset(test_df, tokenizer, is_test=True)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

    model = SentimentModel(cfg.model_name).to(cfg.device)
    fgm = FGM(model)

    crit_sent = nn.CrossEntropyLoss(label_smoothing=0.1)
    crit_topic = nn.CrossEntropyLoss()

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    total_steps = (len(train_loader) // cfg.gradient_accumulation_steps) * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    
    optimizer.zero_grad() # init zero grad

    best_f1 = 0
    for epoch in range(cfg.epochs):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, crit_sent, crit_topic, fgm)
        val_f1 = validate(model, val_loader)
        print(f'Epoch {epoch+1}: loss={loss:.4f} val_f1={val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f'best_fold_{fold}.pt')

    # predict test
    model.load_state_dict(torch.load(f'best_fold_{fold}.pt'))
    model.eval()
    fold_probs = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(cfg.device)
            attention_mask = batch['attention_mask'].to(cfg.device)
            _, sent_logits = model(input_ids, attention_mask)
            probs = torch.softmax(sent_logits, dim=1).cpu().numpy()
            fold_probs.append(probs)

    test_probs.append(np.concatenate(fold_probs, axis=0))

# =========================
# 8. ENSEMBLE + PSEUDO READY
# =========================
mean_probs = np.mean(test_probs, axis=0)
preds = np.argmax(mean_probs, axis=1)

submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': preds
})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')

# pseudo labeling optional
confidence = mean_probs.max(axis=1)
pseudo_mask = confidence > 0.97
pseudo_df = test_df[pseudo_mask].copy()
pseudo_df['sentiment'] = preds[pseudo_mask]
pseudo_df['topic'] = 0
print(f'Pseudo samples: {len(pseudo_df)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.1 MB/s eta 0:00:00


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 3: loss=0.4644 val_f1=0.8284


100%|██████████| 1133/1133 [06:53<00:00,  2.74it/s]


Epoch 4: loss=0.4169 val_f1=0.8235


100%|██████████| 1133/1133 [06:52<00:00,  2.75it/s]


Epoch 5: loss=0.3999 val_f1=0.8261
Saved submission.csv
Pseudo samples: 20
